This journal is used to validate the merged TSV files generated by the collector pipeline's 2_cdhit_merge_fastas.py.

In [33]:
# imports
import os

# choose which file to examine
file_path = "../data/2_merged/merged_HE.tsv"
raw_data_dir = "../data/1_raw/he_proteome/"

# check working directory
print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/keitaichii/Programming/RESEARCH/TORII/tardigrade_proteome_KG/notebooks


In [34]:
# show first few lines of the file
with open(file_path, 'r') as file:
    for _ in range(5):
        line = file.readline()
        print(line.strip())

UniProt_ID	UniProt_Length	NCBI_ID	NCBI_Length	Other_IDs
A0A1W0XE67	365	OQV25661.1	365	OQV25659.1;OQV25660.1;A0A1W0XDW5
A0A9X6RKV7	419	OWA51663.1	419	OWA51664.1;OWA55660.1;A0A9X6RL07
A0A1W0XAQ4	103	OQV20927.1	103	OQV24553.1;OQV24609.1;OQV24682.1;OWA54133.1
A0A1W0X7P1	359	OQV23400.1	359	OQV23401.1;OQV23402.1;A0A1W0X7W1;A0A1W0X811


In [35]:
# basic stats on the file
line_count = 0
with open(file_path, 'r') as f:
    for line in f:
        line_count += 1 # count lines
print(f"Total lines in {file_path}: {line_count}")

# compare to number of lines in files of raw data directory
for filename in os.listdir(raw_data_dir):
    if filename.endswith(".fasta") or filename.endswith(".faa"):
        file_path_raw = os.path.join(raw_data_dir, filename)
        raw_line_count = 0
        with open(file_path_raw, 'r') as f:
            for line in f:
                if line.startswith('>'):  # count only header lines 
                    raw_line_count += 1
        print(f"Total lines in {file_path_raw}: {raw_line_count}")

Total lines in ../data/2_merged/merged_HE.tsv: 20735


FileNotFoundError: [Errno 2] No such file or directory: '../data/1_raw/he_proteome/'

In [36]:
# check for missing values in key columns
key_columns = [0, 1, 2, 3] 
missing_counts = {col: 0 for col in key_columns}
with open(file_path, 'r') as f:
    header = f.readline().strip().split('\t') # read header
    for line in f:
        fields = line.strip().split('\t')
        for col in key_columns:
            if fields[col] == '' or fields[col].lower() == 'na':
                missing_counts[col] += 1
print("Missing values in key columns:")
for col, count in missing_counts.items():
    print(f"Column {header[col]}: {count} missing values")

print()

# show where missing values are located
with open(file_path, 'r') as f:
    header = f.readline().strip().split('\t') # read header
    for line_num, line in enumerate(f, start=2): # start=2 to account for header
        fields = line.strip().split('\t')
        for col in key_columns:
            if fields[col] == '' or fields[col].lower() == 'na':
                print(f"Missing value at line {line_num}, column {header[col]}")

Missing values in key columns:
Column UniProt_ID: 0 missing values
Column UniProt_Length: 0 missing values
Column NCBI_ID: 9 missing values
Column NCBI_Length: 0 missing values

Missing value at line 1296, column NCBI_ID
Missing value at line 20728, column NCBI_ID
Missing value at line 20729, column NCBI_ID
Missing value at line 20730, column NCBI_ID
Missing value at line 20731, column NCBI_ID
Missing value at line 20732, column NCBI_ID
Missing value at line 20733, column NCBI_ID
Missing value at line 20734, column NCBI_ID
Missing value at line 20735, column NCBI_ID


In [37]:
# check values in columns 2 and 4 match (the lengths)
with open(file_path, 'r') as f:
    header = f.readline().strip().split('\t')
    for line_num, line in enumerate(f, start=2):
        fields = line.strip().split('\t')
        length_col = int(fields[1])
        length_faa_col = int(fields[3])
        if length_col != length_faa_col:
            print(f"Length mismatch at line {line_num}: column 2 = {length_col}, column 4 = {length_faa_col}")


# KNOWN ISSUES:
# RV mismatch at line 17874: column 2 = 236, column 4 = 322 > documented on UniProt (Reason: Erroneous initiation Extended N-terminus)
# RV mismatch at line 22417: column 2 = 129, column 4 = 0 > missing sequence in NCBI

Length mismatch at line 119: column 2 = 414, column 4 = 90
Length mismatch at line 1296: column 2 = 227, column 4 = 0
Length mismatch at line 20728: column 2 = 646, column 4 = 0
Length mismatch at line 20729: column 2 = 89, column 4 = 0
Length mismatch at line 20730: column 2 = 264, column 4 = 0
Length mismatch at line 20731: column 2 = 224, column 4 = 0
Length mismatch at line 20732: column 2 = 237, column 4 = 0
Length mismatch at line 20733: column 2 = 227, column 4 = 0
Length mismatch at line 20734: column 2 = 498, column 4 = 0
Length mismatch at line 20735: column 2 = 614, column 4 = 0


In [39]:
# check if ID formats are valid for UniProt
with open(file_path, 'r') as f:
    header = f.readline().strip().split('\t')
    for line_num, line in enumerate(f, start=2):
        fields = line.strip().split('\t')
        uniprot_id = fields[0]
        if not (len(uniprot_id) == 6 or len(uniprot_id) == 10) or not uniprot_id[0].isalpha() or not uniprot_id[1:].isalnum():
            print(f"Invalid UniProt ID format at line {line_num}: {uniprot_id}")